In [ ]:
import os
import sys
os.chdir("/workspaces/dev")
paths = [
    "./modules/python-utils",
    "./modules/ai-utils",
]
for path in paths:
    sys.path.append(os.path.abspath(path))
print(f"Current Python version: {sys.version}")

In [ ]:
from faster_whisper import WhisperModel
from pathlib import Path
import time

In [ ]:
from sj_ai_utils.asr.whisper_utils import *
from sj_ai_utils.datasets.esic_v1 import *
from sj_ai_utils.evaluator.sclite_utils import *
from sj_utils.string_utils import *
from sj_utils.audio_utils import *

In [ ]:
SOURCE = "/workspaces/dev/datasets/ESIC-v1.1/v1.1/test/"
MODEL_SIZE = "large-v3"
SAMPLE_RATE = 16000

In [ ]:
model = WhisperModel(MODEL_SIZE, device="cuda", compute_type="float16")

In [ ]:
src = Path(SOURCE)

In [ ]:
def transcriber(mp4:Path) -> TRNFormat:
    audio, _ = load_audio_from_mp4(mp4, sr=SAMPLE_RATE)

    print(f"Processing")
    print(f"\tAudio name: {mp4.name}")
    print(f"\tAudio length: {len(audio) / SAMPLE_RATE:.2f} seconds")
    start_time = time.perf_counter()
    segments, _= model.transcribe(audio, beam_size=5, language="en")
    end_time = time.perf_counter()
    print(f"Processed time: {end_time - start_time:.2f} seconds")

    text = segments_to_text(segments)

    return normalize_text_only_en(text).upper()

In [ ]:
%%time
data = search_all_ref_and_hyp(
    src, transcriber, lambda x:normalize_text_only_en(x).upper(), 5
)
# CPU times: user 4min 9s, sys: 6.38 s, total: 4min 15s
# Wall time: 50.2 s

In [ ]:
concat_result = {}
for value in data.values():
    for k, v in value.items():
        if k not in concat_result:
            concat_result[k] = []
        concat_result[k].append(v)

In [ ]:
output = sclite_trn(
    concat_result["ref"],
    concat_result["hyp"],
)

In [ ]:
parse_sclite_summary(output)